**Note: Try to avoid *GROUP BY* clause to solve the problems**

For the problems use the *Health Insurance Claim* dataset. You can get the details as well as the dataset from [here](https://www.kaggle.com/datasets/thedevastator/insurance-claim-analysis-demographic-and-health).

### **Problem 1:** What are the top 5 patients who claimed the highest insurance amounts?

### **Problem 2:** What is the average insurance claimed by patients based on the number of children they have?

### **Problem 3:** What is the highest and lowest claimed amount by patients in each region?

### **Problem 4:** What is the percentage of smokers in each age group?

### **Problem 5:** What is the difference between the claimed amount of each patient and the first claimed amount of that patient?

### **Problem 6:** For each patient, calculate the difference between their claimed amount and the average claimed amount of patients with the same number of children.

### **Problem 7:** Show the patient with the highest BMI in each region and their respective rank.

### **Problem 8:** Calculate the difference between the claimed amount of each patient and the claimed amount of the patient who has the highest BMI in their region.

### **Problem 9:** For each patient, calculate the difference in claim amount between the patient and the patient with the highest claim amount among patients with the same bmi and smoker status, within the same region. Return the result in descending order difference.

### **Problem 10:** For each patient, find the maximum BMI value among their next three records (ordered by age).

### **Problem 11:** For each patient, find the rolling average of the last 2 claims.

### **Problem 12:** Find the first claimed insurance value for male and female patients, within each region order the data by patient age in ascending order, and only include patients who are non-diabetic and have a bmi value between 25 and 30.

In [ ]:
# 1.What are the top 5 patients who claimed the highest insurance amounts?
SELECT *
FROM( SELECT * ,
      ROW_NUMBER() OVER( ORDER BY claim DESC ) AS 'rno'
      FROM demodatabase.insurance) t
      WHERE t.rno <6

# 2.What is the average insurance claimed by patients based on the number of children they have?
SELECT DISTINCT children , AVG_claim
FROM(SELECT *,
     AVG(claim) OVER(PARTITION BY children ) AS 'AVG_claim'
     FROM demodatabase.insurance) t

# 3.What is the highest and lowest claimed amount by patients in each region?
SELECT DISTINCT region , MAX_claim,MIN_claim
FROM ( SELECT *,
	   MAX(claim) OVER(PARTITION BY region ) AS 'MAX_claim',
       MIN(claim) OVER(PARTITION BY region) AS 'MIN_claim'
       FROM demodatabase.insurance) t
# 4.What is the percentage of smokers in each age group?
WITH smoker_cnt AS
(SELECT *,
COUNT(smoker) OVER (PARTITION BY age) AS 'smoker_cn'
FROM demodatabase.insurance
) SELECT DISTINCT smoker_cnt.age ,  100 * COUNT(smoker) OVER(PARTITION BY age )/smoker_cnt.smoker_cn AS 'percentage'
  FROM smoker_cnt
  WHERE smoker = 'Yes'
  # alternative and better(gpt)
  WITH smoker_cnt AS (
    SELECT
        age,
        COUNT(*) OVER (PARTITION BY age) AS total_cnt
    FROM demodatabase.insurance
),
smoker_only AS (
    SELECT
        age,
        COUNT(*) OVER (PARTITION BY age) AS smoker_cnt
    FROM demodatabase.insurance
    WHERE smoker = 'Yes'
)
SELECT DISTINCT
       s.age,
       100.0 * s.smoker_cnt / t.total_cnt AS percentage
FROM smoker_only s
JOIN smoker_cnt t
ON s.age = t.age;
# 5.What is the difference between the claimed amount of each patient and the first claimed amount of that patient?
# 6.for each patient, calculate the difference between their claimed amount and the average claimed amount of patients with the same number of children.
SELECT
    PatientID,
    claim - (
        SELECT AVG(claim)
        FROM demodatabase.insurance t2
        WHERE t2.children = t1.children
    ) AS difference
FROM demodatabase.insurance t1;

# 7.Show the patient with the highest BMI in each region and their respective rank.
SELECT PatientID,bmi,region,rank_by_region
FROM( SELECT *,
RANK() OVER(PARTITION BY region ORDER BY bmi DESC) AS 'rank_by_region'
FROM demodatabase.insurance) t
WHERE t.rank_by_region < 2

#8.Calculate the difference between the claimed amount of each patient and the claimed amount of the patient who has the highest BMI in their region.
SELECT
    PatientID,
    claim - (
        SELECT claim
        FROM demodatabase.insurance t2
        WHERE t2.region = t1.region
        ORDER BY bmi DESC
        LIMIT 1
    ) AS difference_bmi
FROM demodatabase.insurance t1;
# 9.For each patient, calculate the difference in claim amount between the patient and the patient with the highest claim amount among patients with the same bmi and smoker status, within the same region. Return the result in descending order difference.

# 10.For each patient, find the maximum BMI value among their next three records (ordered by age).
SELECT
    PatientID,
    age,
    bmi,
    MAX(bmi) OVER (
        PARTITION BY PatientID
        ORDER BY age
        ROWS BETWEEN 1 FOLLOWING AND 3 FOLLOWING
    ) AS max_next_3_bmi
FROM demodatabase.insurance;
# 11.For each patient, find the rolling average of the last 2 claims.

# 12.  Find the first claimed insurance value for male and female patients, within each region order the data by patient age in ascending order, and only include patients who are non-diabetic and have a bmi value between 25 and 30.

SyntaxError: invalid syntax (3592644531.py, line 2)